In [2]:
import os
os.environ["GROQ_API_KEY"] = "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"

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7
)

# Define the expected JSON structure
parser = JsonOutputParser(pydantic_object={
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "price": {"type": "number"},
        "features": {
            "type": "array",
            "items": {"type": "string"}
        }
    }
})

# Create a simple prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract product details into JSON with this structure:
        {{
            "name": "product name here",
            "price": number_here_without_currency_symbol,
            "features": ["feature1", "feature2", "feature3"]
        }}"""),
    ("user", "{input}")
])

# Create the chain that guarantees JSON output
chain = prompt | llm | parser

def parse_product(description: str) -> dict:
    result = chain.invoke({"input": description})
    print(json.dumps(result, indent=2))

        
# Example usage
description = """The Kees Van Der Westen Speedster is a high-end, single-group espresso machine known for its precision, performance, 
and industrial design. Handcrafted in the Netherlands, it features dual boilers for brewing and steaming, PID temperature control for 
consistency, and a unique pre-infusion system to enhance flavor extraction. Designed for enthusiasts and professionals, it offers 
customizable aesthetics, exceptional thermal stability, and intuitive operation via a lever system. The pricing is approximatelyt $14,499 
depending on the retailer and customization options."""

parse_product(description)


{
  "name": "Kees Van Der Westen Speedster",
  "price": 14499,
  "features": [
    "Dual boilers for brewing and steaming",
    "PID temperature control for consistency",
    "Unique pre-infusion system to enhance flavor extraction",
    "Customizable aesthetics",
    "Exceptional thermal stability",
    "Intuitive operation via a lever system"
  ]
}


In [3]:
from langchain_community.llms.huggingface_endpoint import HuggingFaceEndpoint

In [5]:
# Basic Example (no streaming)
llm = HuggingFaceEndpoint(
    endpoint_url="https://huggingface.co/Alibaba-NLP/gte-Qwen1.5-7B-instruct",
    max_new_tokens=512,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_API_TOKEN")
)
print(llm.invoke("What is Deep Learning?"))


/home/voidreaper/Projects/Mini-Project/datamat/env/lib/python3.9/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


ValueError: Task unknown has no recommended model. Please specify a model explicitly. Visit https://huggingface.co/tasks for more info.

In [6]:

# Streaming response example
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

callbacks = [StreamingStdOutCallbackHandler()]
llm = HuggingFaceEndpoint(
    endpoint_url="http://localhost:8010/",
    max_new_tokens=512,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
    callbacks=callbacks,
    streaming=True,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)
print(llm.invoke("What is Deep Learning?"))

/home/voidreaper/Projects/Mini-Project/datamat/env/lib/python3.9/site-packages/huggingface_hub/inference/_client.py:2279: FutureWarning: `stop_sequences` is a deprecated argument for `text_generation` task and will be removed in version '0.28.0'. Use `stop` instead.
  warnings.warn(


ConnectionError: (MaxRetryError("HTTPConnectionPool(host='localhost', port=8010): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x780f1e135eb0>: Failed to establish a new connection: [Errno 111] Connection refused'))"), '(Request ID: 7fc9073b-45a7-4b60-ba53-153a402f35a6)')

In [8]:
KEY = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [9]:
from langchain_community.llms import HuggingFaceHub
hf = HuggingFaceHub(repo_id="Alibaba-NLP/gte-Qwen1.5-7B-instruct", huggingfacehub_api_token=KEY)

/tmp/ipykernel_39129/3913812363.py:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  hf = HuggingFaceHub(repo_id="Alibaba-NLP/gte-Qwen1.5-7B-instruct", huggingfacehub_api_token=KEY)


ValidationError: 1 validation error for HuggingFaceHub
  Value error, Got invalid task sentence-similarity, currently only dict_keys(['translation', 'summarization', 'conversational', 'text-generation', 'text2text-generation']) are supported [type=value_error, input_value={'repo_id': 'Alibaba-NLP/...', 'model_kwargs': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error